# 07 - Reconstructing the Conway--Kim--Politarczyk calculations

This notebook reconstructs the explicit calculations in Anthony Conway, Min Hoon Kim, and Wojciech Politarczyk, *Non-slice linear combinations of iterated torus knots*. Its central example is the four-summand knot

`J(3,2,19,23) = T(3,2;3,19) # -T(3,19) # -T(3,2;3,23) # T(3,23)`.

That particular numerical example is chosen for this tutorial; it is not presented as a distinguished example in the paper. Taking `p=3` makes the character orbit genuinely three-dimensional, while the outer primes `19` and `23` satisfy the algebraic-cabling inequality `q_i > 3*3*2`.

The notebook reproduces the paper's finite, exact ingredients:

1. the `s`-levels from Section 5.1 and their formal cancellations;
2. `H_1(Sigma_p(T(p,q))) = (Z/qZ)^(p-1)` and the character orbit of Lemma 3.1;
3. the metabelian representation matrices of Proposition 3.2;
4. both Fox determinants and the exterior polynomial of Proposition 3.3;
5. the zero-surgery polynomial of Corollary 3.4; and
6. the cyclotomic root supports used to separate Witt-primary contributions.

It does **not** claim to reproduce the complete proof of Theorem 5.1. A full implementation still requires general equivariant linking forms, `Z_p`-invariant metabolizers, graph anti-isometries, and Witt-class metabolicity tests. The final section marks that boundary explicitly.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ, identity_matrix
from gaknot import (
    BranchedCoverHomology,
    Character,
    GeneralizedAlgebraicKnot,
    YanagidaTorusData,
    ckp_torus_knot_data,
    zero_surgery_twisted_alexander_torus_knot,
)
from gaknot.invariants.twisted_alexander import (
    twisted_alexander_torus_knot,
)

## 2. Optional complete text log

Every result produced below is sent through `record`. Set `WRITE_CKP_LOG=True` to save the same labeled values in `computation_logs/conway_kim_politarczyk_reconstruction.txt`. Matrices and root tables are included, so the text file provides a deterministic audit independent of notebook output cells.

The log directory is ignored by Git. The committed notebook therefore remains small, output-free, and reproducible even when a user chooses to retain a detailed local calculation transcript.

In [ ]:
WRITE_CKP_LOG = False
CKP_LOG_PATH = (
    repository_root
    / "computation_logs"
    / "conway_kim_politarczyk_reconstruction.txt"
)

audit_lines = [
    "CONWAY--KIM--POLITARCZYK COMPUTATION RECONSTRUCTION",
    "example=J(3,2,19,23)",
]

def record(label, value):
    """Print one exact result and retain the same text for optional logging."""
    line = f"{label}={value}"
    print(line)
    audit_lines.append(line)
    return value

record("write_log", WRITE_CKP_LOG)
record("log_path", CKP_LOG_PATH)

## 3. A concrete four-summand knot

The package writes each cable sequence from the innermost torus knot to the outermost pattern. Thus `[(3,2),(3,19)]` means the `(3,19)`-cable of `T(3,2)`, denoted `T(3,2;3,19)` in the paper.

The signs are chosen so that the outer `T(3,19)` and `T(3,23)` contributions cancel separately, while the two reparametrized inner `T(3,2)` contributions cancel each other. This is the simplest concrete model of the four-term pattern discussed in the introduction.

In [ ]:
example_knot = GeneralizedAlgebraicKnot([
    (1, [(3, 2), (3, 19)]),
    (-1, [(3, 19)]),
    (-1, [(3, 2), (3, 23)]),
    (1, [(3, 23)]),
])

record("knot", example_knot)
record("description", example_knot.description)
record("number_of_summands", len(example_knot))
record("alexander_polynomial", example_knot.alexander_polynomial())

The full Alexander polynomial is not expected to be one: the knot is a nontrivial formal connected sum. Algebraic sliceness is instead reflected in a level-by-level Witt cancellation. Vanishing of the Levine--Tristram signature is a useful exact consistency check, although signature vanishing alone would not prove algebraic sliceness for an arbitrary knot.

In [ ]:
ordinary_signature = example_knot.signature()

record("ordinary_signature_jump_count",
       len(ordinary_signature.jumps_counter))
record("ordinary_signature_is_zero",
       ordinary_signature.is_zero_everywhere())
record("ordinary_signature_at_minus_one",
       ordinary_signature(QQ(1) / 2))

## 4. The `s`-levels of Section 5.1

For an iterated knot `T(p,q_1;...;p,q_l)`, level `s` selects the torus knot whose second parameter is `q_(l-s)`. A component shorter than `s+1` contributes the unknot. The classical Blanchfield form is then decomposed schematically as

`Bl(K)(t) = direct_sum_s Bl(K_s(K))(t^(p^s))`.

The records below retain every expanded signed term. They also collect coefficients with equal `q`, record the substitution power `p^s`, and list the moduli `p^(s+1)q` of the roots appearing before cancellation.

In [ ]:
levels = example_knot.ckp_cable_levels()
record("number_of_nonempty_levels", len(levels))

for level in levels:
    expanded_terms = tuple(
        (
            int(term.sign),
            int(term.p),
            int(term.q),
            int(term.source_component),
            int(term.source_layer),
            int(term.root_modulus),
        )
        for term in level.terms
    )
    record(f"level_{level.s}_expanded_terms", expanded_terms)
    record(f"level_{level.s}_signed_multiplicities",
           level.signed_multiplicities)
    record(f"level_{level.s}_substitution_power",
           level.substitution_power)
    record(f"level_{level.s}_root_moduli", level.root_moduli)
    record(f"level_{level.s}_is_formally_zero",
           level.is_formally_zero)

assert all(level.is_formally_zero for level in levels)

At level zero the expanded sum is

`T(3,19) # -T(3,19) # -T(3,23) # T(3,23)`.

At level one it is `T(3,2) # -T(3,2)`. Levels `s >= 2` contain only unknots and are omitted. The root moduli at distinct levels are different, illustrating the separation argument in Proposition 5.4 before formal cancellation is applied.

## 5. Branched-cover homology and a character orbit

The outer `T(3,19)` summand controls a 19-primary part of the three-fold branched-cover homology. Proposition 2.4 gives

`H_1(Sigma_3(T(3,19))) = (Z/19Z)^2`.

The public character is entered in the Smith basis as `(1/19,0)`. Lemma 3.1 instead uses the three values on a cyclic deck orbit `x_0,x_1,x_2`, subject to the single relation `x_0+x_1+x_2=0`. The package performs this basis conversion exactly.

In [ ]:
torus_3_19 = GeneralizedAlgebraicKnot.torus_knot(3, 19)
homology_3_19 = BranchedCoverHomology(torus_3_19, 3)
character_19 = Character(
    homology_3_19,
    [[[QQ(1) / 19, QQ(0)]]],
)
data_19 = ckp_torus_knot_data(torus_3_19, character_19)

record("H1_T_3_19", homology_3_19)
record("H1_structural_factors", homology_3_19.invariant_factors)
record("H1_canonical_factors",
       homology_3_19.canonical_invariant_factors)
record("smith_character_values", tuple(character_19.values))
record("deck_orbit_a", data_19.orbit.a_values)
record("deck_orbit_phase_arguments",
       data_19.orbit.phase_arguments)
record("deck_orbit_sum_mod_19",
       sum(data_19.orbit.a_values) % 19)

assert data_19.orbit.a_values == (18, 1, 0)
assert sum(data_19.orbit.a_values) % 19 == 0

The resulting orbit is `(18,1,0)`. Its ordering is not guessed: it comes from the same companion-to-Smith basis change used throughout `gaknot`. Reordering the entries leaves the final denominator product unchanged, but it would change matrices and phase assignments in calculations where the cyclic order matters.

## 6. Proposition 3.2: explicit metabelian matrices

Let `A=A_3(t)` be the cyclic shift matrix. It satisfies `A^3=tI`. For the torus-knot group presentation

`pi_1(E(T(3,19))) = <c_1,c_2 | c_1^3=c_2^19>`,

Proposition 3.2 gives, after conjugation,

`alpha(c_1)=A^19`,

`alpha(c_2)=t*diag(zeta_19^18,zeta_19,1)`.

The following cell prints the literal exact matrices over `Q(zeta_19)(t)` and verifies the group relation rather than assuming it.

In [ ]:
identity_3 = identity_matrix(data_19.function_field, 3)

record("coefficient_field", data_19.coefficient_field)
record("A_3_t", data_19.A)
record("alpha_c1", data_19.c1_image)
record("alpha_c2", data_19.c2_image)
record("A_cubed_equals_tI",
       data_19.A ** 3 == data_19.t * identity_3)
record("c1_cubed_equals_c2_to_19",
       data_19.c1_image ** 3 == data_19.c2_image ** 19)
record("representation_relation_holds", data_19.relation_holds)

assert data_19.A ** 3 == data_19.t * identity_3
assert data_19.relation_holds

## 7. Proposition 3.3: the two Fox determinants

Differentiating the relator `c_1^p c_2^(-q)` with respect to `c_1` produces

`I + alpha(c_1) + ... + alpha(c_1)^(p-1)`.

For `p=3`, this is `I + alpha(c_1) + alpha(c_1)^2`. Its determinant is `(1-t^19)^2`. Deleting the `c_2` column in the Fox formula places `det(alpha(c_2)-I)` in the denominator. The diagonal form of `alpha(c_2)` makes that determinant the ordered product of the three character factors.

In [ ]:
expected_fox_numerator = (1 - data_19.t ** 19) ** 2
expected_fox_denominator = data_19.function_field.one()
for a_value in data_19.orbit.a_values:
    expected_fox_denominator *= (
        data_19.t * data_19.zeta ** a_value - 1
    )

record("fox_numerator_matrix", data_19.fox_numerator_matrix)
record("fox_numerator_determinant",
       data_19.fox_numerator_determinant)
record("expected_fox_numerator", expected_fox_numerator)
record("fox_numerator_formula_holds",
       data_19.fox_numerator_determinant == expected_fox_numerator)
record("fox_denominator_determinant",
       data_19.fox_denominator_determinant)
record("expected_fox_denominator", expected_fox_denominator)
record("fox_denominator_formula_holds",
       data_19.fox_denominator_determinant == expected_fox_denominator)

assert data_19.fox_numerator_determinant == expected_fox_numerator
assert data_19.fox_denominator_determinant == expected_fox_denominator

The exterior twisted Alexander representative is the quotient of those determinants. This is an equality of the exact representatives selected in Proposition 3.3, not merely equality up to a Laurent monomial or a nonzero cyclotomic scalar.

In [ ]:
exterior_polynomial_19 = (
    data_19.fox_numerator_determinant
    / data_19.fox_denominator_determinant
)
legacy_exterior_19 = twisted_alexander_torus_knot(
    torus_3_19, character_19
)

record("exterior_twisted_alexander",
       data_19.exterior_twisted_alexander)
record("fox_quotient_equals_exterior",
       exterior_polynomial_19 == data_19.exterior_twisted_alexander)
record("historical_api_equals_ckp_data",
       legacy_exterior_19 == data_19.exterior_twisted_alexander)

assert exterior_polynomial_19 == data_19.exterior_twisted_alexander
assert legacy_exterior_19 == data_19.exterior_twisted_alexander

## 8. Corollary 3.4: pass from the exterior to zero surgery

For the zero-framed surgery `M_T(p,q)`, the paper's fixed representative is

`(-1)^(p-1) * Delta_exterior(t) / (t-1)`.

The sign is a unit and therefore invisible to the abstract order ideal, but retaining it makes the software reproduce the displayed formula literally. Here `p=3`, so the sign is positive.

In [ ]:
expected_zero_surgery_19 = (
    (-1) ** (data_19.p - 1)
    * data_19.exterior_twisted_alexander
    / (data_19.t - 1)
)
public_zero_surgery_19 = (
    zero_surgery_twisted_alexander_torus_knot(
        torus_3_19, character_19
    )
)

record("zero_surgery_twisted_alexander",
       data_19.zero_surgery_twisted_alexander)
record("zero_surgery_formula_holds",
       data_19.zero_surgery_twisted_alexander
       == expected_zero_surgery_19)
record("zero_surgery_public_api_agrees",
       public_zero_surgery_19 == expected_zero_surgery_19)

assert data_19.zero_surgery_twisted_alexander == expected_zero_surgery_19
assert public_zero_surgery_19 == expected_zero_surgery_19

Yanagida's later formulas use the same global matrices under the names `C`, `X=C^q`, and `Y`. The next cross-check is useful because it joins two independent implementations used elsewhere in the project. We perform it on the smaller `T(2,5)` example: Yanagida's object also constructs the much heavier local pairing matrix, which is irrelevant to this identity and unnecessarily slow over `Q(zeta_19)(t)`. Yanagida writes some linear factors with opposite signs, but the total sign makes his zero-surgery order equal to the CKP representative.

In [ ]:
torus_2_5 = GeneralizedAlgebraicKnot.torus_knot(2, 5)
character_5 = Character(
    BranchedCoverHomology(torus_2_5, 2),
    [[[QQ(1) / 5]]],
)
data_5 = ckp_torus_knot_data(torus_2_5, character_5)
yanagida_5 = YanagidaTorusData(2, 5, data_5.orbit.a_values)

record("cross_check_orbit_5", data_5.orbit.a_values)
record("CKP_A_equals_Yanagida_C",
       data_5.A == yanagida_5.C)
record("CKP_c1_equals_Yanagida_X",
       data_5.c1_image == yanagida_5.X)
record("CKP_c2_equals_Yanagida_Y",
       data_5.c2_image == yanagida_5.Y)
record("zero_surgery_orders_agree_with_Yanagida",
       data_5.zero_surgery_twisted_alexander
       == yanagida_5.zero_surgery_twisted_alexander_order)

assert data_5.orbit.a_values == (4, 1)
assert data_5.A == yanagida_5.C
assert data_5.c1_image == yanagida_5.X
assert data_5.c2_image == yanagida_5.Y
assert (
    data_5.zero_surgery_twisted_alexander
    == yanagida_5.zero_surgery_twisted_alexander_order
)

## 9. Exact cyclotomic root support

The numerator `(1-t^q)^(p-1)` contributes multiplicity `p-1` at every `q`-th root. A denominator factor `t*zeta_q^a-1` removes one copy at `zeta_q^(-a)`. Zero surgery removes one further copy at `t=1`.

The root table records exponent `k`, root order, signed multiplicity, and classification for `zeta_q^k`. A positive multiplicity is a zero, a negative one is a pole, and zero means complete cancellation. This signed convention remains honest for trivial characters, whose displayed representatives need not be polynomials.

In [ ]:
def root_table(root_records):
    """Convert exact root records into a compact printable audit table."""
    return tuple(
        (
            int(root.exponent),
            int(root.root_order),
            int(root.multiplicity),
            root.kind,
        )
        for root in root_records
    )

exterior_root_table_19 = root_table(
    data_19.exterior_root_multiplicities
)
zero_surgery_root_table_19 = root_table(
    data_19.zero_surgery_root_multiplicities
)

record("exterior_root_table_19", exterior_root_table_19)
record("zero_surgery_root_table_19",
       zero_surgery_root_table_19)
record("exterior_zero_exponents_19",
       tuple(int(root.exponent)
             for root in data_19.exterior_zero_support))
record("zero_surgery_zero_exponents_19",
       tuple(int(root.exponent)
             for root in data_19.zero_surgery_zero_support))
record("zero_surgery_poles_19",
       root_table(data_19.zero_surgery_pole_support))

Because the orbit `(18,1,0)` contains one zero, the denominator removes one copy of `t-1`; the additional surgery denominator removes the other. Thus `t=1` is completely cancelled in the zero-surgery representative. Every remaining zero is a primitive 19th root. This is exactly the prime-power root behavior used to separate the twisted part from classical Alexander factors in Section 5.2.2.

## 10. Repeat the root calculation for the 23-primary summand

The second outer prime is handled identically. In the package's computed Smith basis, the values `(1/23,0)` give the deck orbit `(1,0,22)`. Its zero-surgery polynomial is supported at primitive 23rd roots, disjoint from the primitive 19th-root support above. The difference from the cyclic ordering seen for `q=19` is a useful reminder that the Smith-to-companion basis conversion must be computed rather than guessed.

In [ ]:
torus_3_23 = GeneralizedAlgebraicKnot.torus_knot(3, 23)
homology_3_23 = BranchedCoverHomology(torus_3_23, 3)
character_23 = Character(
    homology_3_23,
    [[[QQ(1) / 23, QQ(0)]]],
)
data_23 = ckp_torus_knot_data(torus_3_23, character_23)

root_orders_19 = tuple(sorted({
    int(root.root_order)
    for root in data_19.zero_surgery_zero_support
}))
root_orders_23 = tuple(sorted({
    int(root.root_order)
    for root in data_23.zero_surgery_zero_support
}))

record("H1_T_3_23", homology_3_23)
record("deck_orbit_23", data_23.orbit.a_values)
record("zero_surgery_root_table_23",
       root_table(data_23.zero_surgery_root_multiplicities))
record("zero_surgery_root_orders_19", root_orders_19)
record("zero_surgery_root_orders_23", root_orders_23)
record("root_order_supports_are_disjoint",
       set(root_orders_19).isdisjoint(root_orders_23))

assert data_23.orbit.a_values == (1, 0, 22)
assert root_orders_19 == (19,)
assert root_orders_23 == (23,)
assert set(root_orders_19).isdisjoint(root_orders_23)

There are two related separation mechanisms in the proof:

- classical level terms have roots whose moduli contain different powers of `p`, recorded earlier as `57`, `69`, and `18`; and
- the metabelian zero-surgery orders associated to the selected characters have roots of prime orders `19` and `23`.

Distinct root support lets the Witt-group argument isolate primary summands. The computation verifies the supports; the theorem that metabolicity splits over disjoint supports is mathematical input from the paper, not something inferred numerically here.

## 11. Write the optional audit file

This final cell writes exactly the labeled lines accumulated above. With logging disabled it performs no file-system write. Rerunning the notebook from the beginning produces the same ordered transcript because all calculations use exact rings and deterministic basis conventions.

In [ ]:
record("all_levels_formally_zero",
       all(level.is_formally_zero for level in levels))
record("section_3_matrix_relation_verified",
       data_19.relation_holds and data_23.relation_holds)
record("full_theorem_5_1_certified_by_notebook", False)

if WRITE_CKP_LOG:
    CKP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CKP_LOG_PATH.write_text(
        "\n".join(audit_lines) + "\n",
        encoding="utf-8",
    )
    print("Wrote complete CKP audit to:", CKP_LOG_PATH)
else:
    print("Text logging is disabled; no log file was written.")

## 12. What remains for a complete reconstruction

The calculations above establish the computational foundation, but the main theorem quantifies over **every** `Z_p`-invariant metabolizer. Completing that proof in software requires four further layers:

1. construct the equivariant linking form on the general `p`-fold cover and expose its deck action;
2. represent invariant metabolizers, including the graph-metabolizer case and its anti-isometry;
3. implement the three-case character construction from Lemma 5.9; and
4. assemble signed satellite Blanchfield Witt classes and certify nonmetabolicity on the appropriate root-primary summand.

Until those steps are implemented, the `False` value logged above is deliberate: this notebook reconstructs Propositions 3.2--3.3, Corollary 3.4, the `s`-levels, and the exact root-support calculations, but it does not advertise a computer proof of Theorem 5.1.

## 13. Reference and exercises

Primary reference:

- Anthony Conway, Min Hoon Kim, and Wojciech Politarczyk, *Non-slice linear combinations of iterated torus knots*, Algebraic & Geometric Topology **23** (2023), no. 2, 765--802; [journal](https://doi.org/10.2140/agt.2023.23.765), [arXiv:1910.01368](https://arxiv.org/abs/1910.01368). See especially Proposition 2.4, Lemma 3.1, Propositions 3.2--3.3, Corollary 3.4, Proposition 5.4, Proposition 5.8, and Lemma 5.9.

Exercises:

1. Replace the Smith character `(1/19,0)` by `(0,1/19)`, recompute its orbit, and identify which 19th roots cancel from the denominator.
2. Construct the trivial character on `T(3,19)` and inspect the negative multiplicities at `t=1`; explain why retaining poles is safer than calling every displayed representative a polynomial.
3. Replace `19` and `23` by two other admissible outer primes greater than `18`, and verify both level cancellation and disjoint prime-order support.
4. Turn on logging, rerun the notebook, and check the two Fox determinant identities directly from the stored matrices.